# 🔬 Model 04 — Station Hour AQI Prediction
**AirVintage Production ML Pipeline — Deployment-Ready**

| Input | Source | Free? |
|-------|--------|-------|
| `PM2.5`, `PM10` | OpenAQ / CPCB API | ✅ |
| `NO2`, `SO2`, `CO`, `O3` | OpenAQ / CPCB API | ✅ |
| `Station ID` | CPCB station list | ✅ |
| `Datetime` | System clock | ✅ |

**Dropped:** `NO`, `NOx`, `NH3`, `Benzene`, `Toluene`, `Xylene`

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor, Pool
import joblib

from sklearn.model_selection import train_test_split
from airvintage_ml import (
    aqi_to_bucket, health_advisory,
    add_temporal_features,
    encode_categorical, fillna_production, add_entity_stats,
    compute_metrics, print_metrics_table,
    plot_actual_vs_pred, plot_residuals,
    plot_feature_importance, update_model_registry,
)

DATASET='station_hour_cleaned.csv'; MODEL_KEY='station_hour'
MODELS_DIR='models'; REGISTRY=f'{MODELS_DIR}/model_registry.json'; SEED=42
os.makedirs(MODELS_DIR, exist_ok=True)

CORE_POLLUTANTS = ['PM2.5','PM10','NO2','SO2','CO','O3']
print('CatBoost', __import__('catboost').__version__, '| Inputs:', CORE_POLLUTANTS)

In [ ]:
# Load with float32 to handle 260MB file
dtypes = {c:'float32' for c in CORE_POLLUTANTS + ['AQI']}
df = pd.read_csv(DATASET, parse_dates=['Datetime'], dtype=dtypes, low_memory=True)
print(f'Shape: {df.shape[0]:,} x {df.shape[1]} | Stations: {df["StationId"].nunique()}')
print(f'Memory: {df.memory_usage(deep=True).sum()/1e6:.0f} MB')
df.head(3)

In [ ]:
# ── EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

df_tmp = df.copy()
df_tmp['Hour'] = pd.to_datetime(df_tmp['Datetime']).dt.hour

h = df_tmp.groupby('Hour')['AQI'].agg(['mean','std'])
axes[0,0].fill_between(h.index, h['mean']-h['std'], h['mean']+h['std'], alpha=0.15, color='#f12711')
axes[0,0].plot(h.index, h['mean'], 'o-', color='#f12711', lw=2.5, ms=5)
axes[0,0].set_title('Station Hourly AQI +/- 1sigma', fontweight='bold')
axes[0,0].set_xticks(range(0,24,2))

axes[0,1].hist(df['AQI'].dropna(), bins=80, color='#f093fb', edgecolor='white', alpha=0.85)
axes[0,1].set_title('AQI Distribution', fontweight='bold')

avail = [c for c in CORE_POLLUTANTS if c in df.columns]
corr = df[avail+['AQI']].corr()['AQI'].drop('AQI').abs().sort_values()
axes[0,2].barh(corr.index, corr.values, color='#4facfe')
axes[0,2].set_title('Core Pollutant Correlation with AQI', fontweight='bold')

stn_mean = df.groupby('StationId')['AQI'].mean()
top = stn_mean.sort_values(ascending=False).head(10)
axes[1,0].barh(top.index[::-1], top.values[::-1], color=plt.cm.Reds(np.linspace(0.4,0.9,10)))
axes[1,0].set_title('Top 10 Polluted Stations', fontweight='bold')
axes[1,0].tick_params(axis='y', labelsize=7)

axes[1,1].hist(stn_mean, bins=40, color='#667eea', edgecolor='white', alpha=0.85)
axes[1,1].set_title('Station Mean AQI Distribution', fontweight='bold')

df[[c for c in CORE_POLLUTANTS if c in df.columns]].describe().T[['mean','std','min','max']].plot.bar(ax=axes[1,2])
axes[1,2].set_title('Core Pollutant Stats', fontweight='bold')
axes[1,2].tick_params(axis='x', rotation=30)

fig.suptitle('Station Hour EDA — Production Feature Set', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/eda_04_station_hour.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Engineering — Production Only
df_feat = df.copy()
df_feat = add_temporal_features(df_feat, 'Datetime')

df_feat['PM_ratio'] = df_feat['PM2.5'] / (df_feat['PM10'].astype(float) + 1e-6)
df_feat['Total_PM'] = df_feat['PM2.5'].astype(float) + df_feat['PM10'].astype(float)
df_feat['NO2_SO2']  = df_feat['NO2'].astype(float)   + df_feat['SO2'].astype(float)
df_feat['CO_O3']    = df_feat['CO'].astype(float)    * df_feat['O3'].astype(float)

# CatBoost handles StationId as a native categorical
df_feat['StationId'] = df_feat['StationId'].astype(str)
df_feat, le_stn   = encode_categorical(df_feat, 'StationId')
df_feat, stn_stats = add_entity_stats(df_feat, 'StationId_encoded', 'AQI')

CAT_FEATURES = ['StationId']
DROP = ['Datetime','AQI','AQI_Bucket',
        'NO','NOx','NH3','Benzene','Toluene','Xylene']
NUM_FEATURES = [c for c in df_feat.columns if c not in DROP + CAT_FEATURES]
FEATURE_COLS = CAT_FEATURES + NUM_FEATURES
CAT_IDX      = [0]  # StationId is at index 0

df_clean = df_feat.dropna(subset=['AQI']).copy()
df_clean[NUM_FEATURES] = df_clean[NUM_FEATURES].fillna(df_clean[NUM_FEATURES].median())
df_clean['StationId']  = df_clean['StationId'].astype(str).fillna('UNKNOWN')

X, y = df_clean[FEATURE_COLS], df_clean['AQI']
print(f'Production features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
X_train,X_tmp,y_train,y_tmp = train_test_split(X,y,test_size=0.30,random_state=SEED)
X_val,X_test,y_val,y_test   = train_test_split(X_tmp,y_tmp,test_size=0.50,random_state=SEED)

train_pool = Pool(X_train, y_train, cat_features=CAT_IDX)
val_pool   = Pool(X_val,   y_val,   cat_features=CAT_IDX)
test_pool  = Pool(X_test,  y_test,  cat_features=CAT_IDX)
print(f'Train:{X_train.shape[0]:,} | Val:{X_val.shape[0]:,} | Test:{X_test.shape[0]:,}')

In [ ]:
model = CatBoostRegressor(
    iterations=2500, learning_rate=0.03, depth=8,
    l2_leaf_reg=3, min_data_in_leaf=20, subsample=0.8,
    colsample_bylevel=0.8, loss_function='RMSE', eval_metric='RMSE',
    random_seed=SEED, early_stopping_rounds=100,
    task_type='CPU', verbose=300
)
model.fit(train_pool, eval_set=val_pool, use_best_model=True)
print(f'Best iteration: {model.best_iteration_}')

In [ ]:
train_m=compute_metrics(y_train,model.predict(Pool(X_train,cat_features=CAT_IDX)))
val_m  =compute_metrics(y_val,  model.predict(Pool(X_val,  cat_features=CAT_IDX)))
test_m =compute_metrics(y_test, model.predict(test_pool))
print_metrics_table({'Train':train_m,'Val':val_m,'Test':test_m})

plot_actual_vs_pred(y_test,model.predict(test_pool),'Actual vs Pred — Station Hour',f'{MODELS_DIR}/avp_04_station_hour.png')
plot_residuals(y_test,model.predict(test_pool),'Residuals — Station Hour',f'{MODELS_DIR}/res_04_station_hour.png')
plot_feature_importance(FEATURE_COLS,model.get_feature_importance(),'Feature Importance — Station Hour',savepath=f'{MODELS_DIR}/fi_04_station_hour.png')

In [ ]:
model.save_model(f'{MODELS_DIR}/04_station_hour_catboost.cbm')
joblib.dump(le_stn, f'{MODELS_DIR}/04_station_hour_le_stn.pkl')
stn_stats.to_csv(f'{MODELS_DIR}/04_station_hour_stn_stats.csv', index=False)
with open(f'{MODELS_DIR}/04_station_hour_features.json','w') as f:
    json.dump({'feature_cols':FEATURE_COLS,'cat_features':CAT_FEATURES,'cat_idx':CAT_IDX},f,indent=2)
with open(f'{MODELS_DIR}/04_station_hour_input_schema.json','w') as f:
    json.dump({'required_inputs':['station_id','datetime']+CORE_POLLUTANTS,
               'note':'All pollutants from OpenAQ/CPCB free API'},f,indent=2)

update_model_registry(REGISTRY,MODEL_KEY,'CatBoost',DATASET,test_m,
    model_paths={'model':f'{MODELS_DIR}/04_station_hour_catboost.cbm',
                 'encoder':f'{MODELS_DIR}/04_station_hour_le_stn.pkl',
                 'stn_stats':f'{MODELS_DIR}/04_station_hour_stn_stats.csv',
                 'features':f'{MODELS_DIR}/04_station_hour_features.json'},
    feature_count=len(FEATURE_COLS),notes='Inputs: PM2.5,PM10,NO2,SO2,CO,O3 + station_id + datetime')
print('Model 04 saved.')

In [ ]:
def predict_station_hour(
    station_id: str, datetime_str: str,
    pm25: float, pm10: float,
    no2: float, so2: float, co: float, o3: float
) -> dict:
    """
    Predict hourly AQI for a monitoring station.
    All inputs available from CPCB/OpenAQ free API.
    """
    row = pd.DataFrame([{'StationId':station_id,'Datetime':datetime_str,
                          'PM2.5':pm25,'PM10':pm10,'NO2':no2,'SO2':so2,'CO':co,'O3':o3}])
    row['Datetime'] = pd.to_datetime(row['Datetime'])
    row = add_temporal_features(row,'Datetime')
    row['PM_ratio']=pm25/(pm10+1e-6); row['Total_PM']=pm25+pm10
    row['NO2_SO2']=no2+so2; row['CO_O3']=co*o3
    row['StationId'] = str(station_id)

    s_enc = int(le_stn.transform([station_id])[0]) if station_id in le_stn.classes_ else -1
    row['StationId_encoded'] = s_enc
    sr = stn_stats[stn_stats['StationId_encoded']==s_enc]
    for col in [c for c in stn_stats.columns if 'AQI' in c]:
        row[col] = float(sr[col].values[0]) if len(sr)>0 else 150.0

    num_feats = [f for f in FEATURE_COLS if f != 'StationId']
    row[num_feats] = row[num_feats].fillna(0)
    pool = Pool(row[FEATURE_COLS], cat_features=CAT_IDX)
    aqi  = float(model.predict(pool)[0])
    b    = aqi_to_bucket(aqi)
    return {'model':'station_hour','station_id':station_id,'datetime':datetime_str,
            'predicted_aqi':round(aqi,2),'aqi_category':b,'health_advisory':health_advisory(b)}

demo = predict_station_hour(df['StationId'].iloc[0],'2025-01-15 17:30:00',
                             pm25=130, pm10=190, no2=70, so2=18, co=2.2, o3=15)
print('Demo:'); [print(f'  {k}: {v}') for k,v in demo.items()]